# ETL Development

This notebook is used to prototype and validate data transformations before implementing them in the production ETL pipeline.

### Load the Dataset

In [4]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/raw/unicorn_companies.csv")

df = pd.read_csv(DATA_PATH)

df.head()

,Company,Valuation,Date Joined,Industry,City,Country,Continent,Year Founded,Funding,Select Investors
0,Bytedance,$180B,4/7/2017,Artificial intelligence,Beijing,China,Asia,2012,$8B,"Sequoia Capital China, SIG Asia Investments, S..."
1,SpaceX,$100B,12/1/2012,Other,Hawthorne,United States,North America,2002,$7B,"Founders Fund, Draper Fisher Jurvetson, Rothen..."
2,SHEIN,$100B,7/3/2018,E-commerce & direct-to-consumer,Shenzhen,China,Asia,2008,$2B,"Tiger Global Management, Sequoia Capital China..."
3,Stripe,$95B,1/23/2014,Fintech,San Francisco,United States,North America,2010,$2B,"Khosla Ventures, LowercaseCapital, capitalG"
4,Klarna,$46B,12/12/2011,Fintech,Stockholm,Sweden,Europe,2005,$4B,"Institutional Venture Partners, Sequoia Capita..."


### Create a Working Copy

In [5]:
work_df = df.copy()

### Standardize Column Names

In [6]:
work_df.columns

Index(['Company', 'Valuation', 'Date Joined', 'Industry', 'City', 'Country',
       'Continent', 'Year Founded', 'Funding', 'Select Investors'],
      dtype='object')

In [7]:
work_df.columns = (
    work_df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
)

work_df.columns

Index(['company', 'valuation', 'date_joined', 'industry', 'city', 'country',
       'continent', 'year_founded', 'funding', 'select_investors'],
      dtype='object')

### Investigate Missing Values

In [8]:
work_df.isna().sum()

company              0
valuation            0
date_joined          0
industry             0
city                16
country              0
continent            0
year_founded         0
funding              0
select_investors     1
dtype: int64

### Handle Missing Values

In [9]:
work_df["city"] = (
    work_df["city"]
    .fillna("Unknown")
)

work_df["select_investors"] = (
    work_df["select_investors"]
    .fillna("Unknown")
)

In [10]:
work_df.isna().sum()

company             0
valuation           0
date_joined         0
industry            0
city                0
country             0
continent           0
year_founded        0
funding             0
select_investors    0
dtype: int64

### Convert Date Column

In [11]:
work_df["date_joined"].dtype

dtype('O')

In [12]:
work_df["date_joined"] = pd.to_datetime(
    work_df["date_joined"]
)

In [13]:
work_df["date_joined"].dtype

dtype('<M8[ns]')

### Create Joined Year Feature

In [14]:
work_df["joined_year"] = (
    work_df["date_joined"]
    .dt.year
)

In [15]:
work_df[
    ["date_joined", "joined_year"]
].head()

,date_joined,joined_year
0,2017-04-07,2017
1,2012-12-01,2012
2,2018-07-03,2018
3,2014-01-23,2014
4,2011-12-12,2011


### Create Company Age Feature

In [16]:
CURRENT_YEAR = 2026

work_df["company_age"] = (
    CURRENT_YEAR
    - work_df["year_founded"]
)

In [17]:
work_df[
    ["company", "year_founded", "company_age"]
].head()

,company,year_founded,company_age
0,Bytedance,2012,14
1,SpaceX,2002,24
2,SHEIN,2008,18
3,Stripe,2010,16
4,Klarna,2005,21


### Investigate Valuation Values

In [18]:
work_df["valuation"].sample(
    20,
    random_state=42
)

542     $2B
370     $2B
307     $3B
493     $2B
350     $3B
237     $4B
475     $2B
578     $2B
462     $2B
978     $1B
545     $2B
713     $1B
141     $5B
275     $3B
428     $2B
453     $2B
808     $1B
361     $2B
1073    $1B
737     $1B
Name: valuation, dtype: object

### Build Conversion Function

In [19]:
def money_to_billions(value):

    if pd.isna(value):
        return None

    value = value.replace("$", "")

    if "B" in value:
        return float(
            value.replace("B", "")
        )

    if "M" in value:
        return (
            float(
                value.replace("M", "")
            ) / 1000
        )

    return None

In [20]:
money_to_billions("$180B")

180.0

In [21]:
money_to_billions("$500M")

0.5

### Convert Valuation

In [22]:
work_df["valuation_billion"] = (
    work_df["valuation"]
    .apply(money_to_billions)
)

In [23]:
work_df[
    [
        "valuation",
        "valuation_billion"
    ]
].head()

,valuation,valuation_billion
0,$180B,180.0
1,$100B,100.0
2,$100B,100.0
3,$95B,95.0
4,$46B,46.0


### Convert Funding

In [24]:
work_df["funding_billion"] = (
    work_df["funding"]
    .apply(money_to_billions)
)

In [25]:
work_df[
    [
        "funding",
        "funding_billion"
    ]
].head()

,funding,funding_billion
0,$8B,8.0
1,$7B,7.0
2,$2B,2.0
3,$2B,2.0
4,$4B,4.0


### Check Duplicates

In [26]:
work_df.duplicated().sum()

np.int64(0)

### Final Validation

In [27]:
work_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1074 entries, 0 to 1073
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   company            1074 non-null   object        
 1   valuation          1074 non-null   object        
 2   date_joined        1074 non-null   datetime64[ns]
 3   industry           1074 non-null   object        
 4   city               1074 non-null   object        
 5   country            1074 non-null   object        
 6   continent          1074 non-null   object        
 7   year_founded       1074 non-null   int64         
 8   funding            1074 non-null   object        
 9   select_investors   1074 non-null   object        
 10  joined_year        1074 non-null   int32         
 11  company_age        1074 non-null   int64         
 12  valuation_billion  1074 non-null   float64       
 13  funding_billion    1062 non-null   float64       
dtypes: datet

In [28]:
work_df.isna().sum()

company               0
valuation             0
date_joined           0
industry              0
city                  0
country               0
continent             0
year_founded          0
funding               0
select_investors      0
joined_year           0
company_age           0
valuation_billion     0
funding_billion      12
dtype: int64

In [33]:
work_df["industry"] = work_df["industry"].str.title()

In [34]:
work_df.head()

,company,valuation,date_joined,industry,city,country,continent,year_founded,funding,select_investors,joined_year,company_age,valuation_billion,funding_billion
0,Bytedance,$180B,2017-04-07,Artificial Intelligence,Beijing,China,Asia,2012,$8B,"Sequoia Capital China, SIG Asia Investments, S...",2017,14,180.0,8.0
1,SpaceX,$100B,2012-12-01,Other,Hawthorne,United States,North America,2002,$7B,"Founders Fund, Draper Fisher Jurvetson, Rothen...",2012,24,100.0,7.0
2,SHEIN,$100B,2018-07-03,E-Commerce & Direct-To-Consumer,Shenzhen,China,Asia,2008,$2B,"Tiger Global Management, Sequoia Capital China...",2018,18,100.0,2.0
3,Stripe,$95B,2014-01-23,Fintech,San Francisco,United States,North America,2010,$2B,"Khosla Ventures, LowercaseCapital, capitalG",2014,16,95.0,2.0
4,Klarna,$46B,2011-12-12,Fintech,Stockholm,Sweden,Europe,2005,$4B,"Institutional Venture Partners, Sequoia Capita...",2011,21,46.0,4.0
